# Libraries and preamble

In [ ]:
install.packages("faraway")

In [ ]:
library(faraway)

In [ ]:
library(tidyverse)

In [ ]:
library(tidyr)

In [ ]:
library(dplyr)

# F4.2

## F4.2.a

In [ ]:
teengamb_base_lm <- lm (
    gamble ~ .,
    data = teengamb
)

In [ ]:
summary(teengamb_base_lm)

In [ ]:
head(teengamb)

In [ ]:
teengamb_male_average <- teengamb %>% 
    filter(sex == 0) %>%
    summarise(across(c(verbal, income, status), \(x) mean(x, na.rm = TRUE))) %>%
    as.data.frame()

In [ ]:
teengamb_male_average$sex = 0

In [ ]:
# Make predictions on new data X_test
teengamb_male_average_prediction <- 
    predict(
        teengamb_base_lm, 
        newdata = teengamb_male_average,
        interval = "prediction", 
        level = 0.95
)


In [ ]:
teengamb_male_average_prediction

Our prediction for the average male is 29.775 and the CI interval at 0.95 is (-16826, 76.37649) which is non sensible.

## F4.2.b

In [ ]:
teengamb_male_max <- teengamb %>% 
    summarise(across(everything(), max))

In [ ]:
teengamb_male_max_prediction <-
    predict(
        teengamb_base_lm, 
        newdata = teengamb_male_max,
        interval = "prediction", 
        level = 0.95
)

In [ ]:
teengamb_male_max_prediction

For males with the highest values in predictors, the dependent value is 49.1896120726776 and the CI at 0.95 is (-9.25, 107.63) which too is non-sensical.

## F4.2.c

In [ ]:
teengamb_sqrt_lm <- 
    lm(sqrt(gamble) ~ ., data = teengamb)

In [ ]:
summary(teengamb_sqrt_lm)

Note from the student: The question says 'The Individual' here is ambiguous. So, I will do it for both the average and the maximum case.

### The sqrt prediction for the teengamb male average is:

In [ ]:
teengamb_sqrt_male_max_prediction <-
    predict(
        teengamb_sqrt_lm, 
        newdata = teengamb_male_max,
        interval = "prediction", 
        level = 0.95
)

In [ ]:
teengamb_male_max_prediction

The predicted value of teengamb for a male with maximum predictors is 49.18. The CI intervals with 95% confidence are (-9.25,107.62) The negative is steill not the most sensible.

### The sqrt prediction for the teengamb male max is:

In [ ]:
teengamb_sqrt_male_average_prediction <-
    predict(
        teengamb_sqrt_lm, 
        newdata = teengamb_male_average,
        interval = "prediction", 
        level = 0.95
)

In [ ]:
teengamb_male_average_prediction

The predicted value of teengamb for a male with average predictors is 29.775. The CI intervals with 95% confidence are (-16.826, 76.376). The negative lower bound is not physically meaningful.

### F4.2.c

In [ ]:
teengamb_female_data_Frame <- 
    data.frame(
        sex = 1,
        income = 1,
        verbal = 10,
        status = 20
    )

In [ ]:
teengamb_female_sqrt_prediction <- 
    predict(
        teengamb_sqrt_lm, 
        newdata = teengamb_female_data_Frame,
        interval = "prediction",
        level = 0.95
    )

In [ ]:
teengamb_female_sqrt_prediction

The prediction with a negative value is not physically sensible. The upper bound is positive but the lower bound is negative.

# F4.3

## F4.3.a

(leaving this here for my own edification)

The 4 was what confused me, turns out that is just the count per group hard coded in.

In [49]:
xtabs(water ~ temp + humid, snail)/4

    humid
temp    45    75   100
  20 72.50 81.50 97.00
  30 69.50 78.25 97.75

We can also do the above as follows using the `tidyverse` api.

In [48]:
snail %>%
  group_by(temp, humid) %>%
  summarize(mean_water = mean(water, na.rm = TRUE), .groups = "drop") %>%
  pivot_wider(names_from = humid, values_from = mean_water)

temp,45,75,100
<int>,<dbl>,<dbl>,<dbl>
20,72.5,81.50,97.00
30,69.5,78.25,97.75


### Can I use the table to predict the water context for a temperature for *$25^{\circ}C$ *and a humidity of 60%?

Short answer: No.

The table represents is a discrete summary of the data at hand. It does address parts of the data in a modular way but that is not nearly enough to make predictions about the data at hand which is continuous in nature.

Maybe ...

I think my answer to this question is that this is something that can be done but also something that should not be done. The data is continuous and moreover wehave access to linear regression rather conviniently and we should use that instead. 


Back of the envelope calculations:

$25^{\circ}C$ is $20^{\circ}C$ + $30^{\circ}C$ divided by 2 perhaps the water content is somewhere between 72.5 and 69.5 or (69.5 + 72.5)/2 = 71

And 60% humiditity is 45 + 75 divided by 2 = 60, so perhaps the water content is somewhere between 72.5 and 81.50 or (72.5 + 81.5)/2 = 77.

And if we were to assign equal weight to temperature and humidity we get (71 + 77)/2 = 74.

Not saying this is scientific at all - just intuitive.

## 4.3.b

In [47]:
water_content_lm <- lm(
    water ~ temp + humid,
    data = snail
)

summary(water_content_lm)


Call:
lm(formula = water ~ temp + humid, data = snail)

Residuals:
    Min      1Q  Median      3Q     Max 
-12.456  -2.915   1.461   3.613   8.749 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) 52.61081    6.85346   7.677 1.59e-07 ***
temp        -0.18333    0.22645  -0.810    0.427    
humid        0.47349    0.05036   9.403 5.63e-09 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 5.547 on 21 degrees of freedom
Multiple R-squared:  0.8092,	Adjusted R-squared:  0.791 
F-statistic: 44.53 on 2 and 21 DF,  p-value: 2.793e-08


In [44]:
predict(
    water_content_lm,
    newdata = data.frame(
        temp = 25,
        humid = 60
    )
)

1 
76.43681

The value comes out to be 76.4368131868132.

My back of the hand-calculations were 74 units and according to linear regression the answer is 76.436. A little off but not sure by how much.

## 4.3.c

Prediction for 30 degrees celcius and 75% humidity.

In [46]:
predict(
    water_content_lm,
    newdata = data.frame(
        temp = 30,
        humid = 75
    )
)

1 
82.62248

The value comes out to be 82.622.

### Merits of the predictions for predictions from B and C i.e. the predictions that use the linear model

They use linear regression to make predictions about the continuous dependent variable which is the right thing to do.

## 4.3.d

By the definition of the linear model and its intercept, the intercept is the expected value of the response variable when all the predictors are equal to zero.

The (0,0) is not quite unique because there are any combinatinons of the predictor pairs that can cancel each other out.

## 4.3.e

The linear model evalutes to:

$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \epsilon$

Where the relevant values are:

$\beta_0 = 52.6$

$\beta_1 = -0.18$

$\beta_2 = 0.47$

The question is bascially asking

humidity = (80 - 52.16 + 0.18 * 25 ) / 0.47

and the answer comes out to be 67.85.

Let's cross-verify with `predict()`.

In [55]:
predict(
    water_content_lm,
    newdata = data.frame(temp = 25, humid = 67.85)
)

1 
80.1537

Which evaluated to `80.15`, I am a tad bit confused by the imprescision of the answer but it is close enough.

# F4.4

The `mdeaths` dataset .

# Scratch